# Notebook 1 — Setup & Dataset Validation
**Source:** `sdp-preprocessing` preprocessed `.npy` dataset  
**Preprocessed path:** `/kaggle/input/datasets/assualttger/sdp-preprocessing/preprocessed/`  
**Structure:** `{ff, celeb, gan}/{rgb, fft, hist}/*.npy` + `metadata_{train,val,test}.json`

**This notebook:**
1. Installs packages  
2. Validates the preprocessed `.npy` files  
3. Builds `manifest.json` (train/val/test splits)  
4. Writes `model_config.yaml` and `model.py`

> Enable **T4 GPU** and **Internet** in Kaggle Settings before running.

In [ ]:
# ── CELL 1: GPU Check ─────────────────────────────────────────
import torch, subprocess

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── CELL 2: Install Packages ──────────────────────────────────
import subprocess, sys

packages = [
    "timm>=0.9.12", "torchcam>=0.3.2", "safetensors>=0.4.3",
    "scikit-learn>=1.4.0", "opencv-python-headless",
    "matplotlib", "seaborn", "pyyaml", "tqdm", "torchvision>=0.17.0",
]

for pkg in packages:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], capture_output=True)
    print(f"  {'✅' if r.returncode == 0 else '❌'} {pkg}")

In [ ]:
# ── CELL 3: Preprocessed Dataset Paths ───────────────────────
# All data comes from the sdp-preprocessing dataset.
# Celeb-DF is ONLY used as a cross-dataset test (never trained on).

PREP = "/kaggle/input/datasets/assualttger/sdp-preprocessing/preprocessed"
FF    = f"{PREP}/ff"
GAN   = f"{PREP}/gan"
CELEB = f"{PREP}/celeb"    # test-only (cross-dataset benchmark)

import os
for name, path in [("ff", FF), ("gan", GAN), ("celeb", CELEB)]:
    n = sum(len(f) for _, _, f in os.walk(path)) if os.path.exists(path) else 0
    icon = "✅" if n > 0 else "❌"
    print(f"  {icon} {name}: {path}  ({n:,} files)")

In [ ]:
# ── CELL 4: Validate .npy File Shapes ────────────────────────
# Each folder has: rgb/*.npy (224,224,3) uint8
#                  fft/*.npy (224,224,3) uint8
#                  hist/*.npy (N,) float32

import numpy as np
from glob import glob

for name, path in [("FF++", FF), ("GAN", GAN), ("Celeb-DF", CELEB)]:
    print(f"\n[{name}]")
    for sub in ["rgb", "fft", "hist"]:
        files = glob(f"{path}/{sub}/*.npy")
        if files:
            sample = np.load(files[0])
            print(f"  {sub}/: {len(files):,} files  shape={sample.shape}  dtype={sample.dtype}")
        else:
            print(f"  {sub}/: NOT FOUND")

In [ ]:
# ── CELL 5: Load Metadata & Build manifest.json ───────────────
# Merges FF++ + GAN for train/val. Celeb-DF is cross-dataset test only.

import json, random, os
random.seed(42)

def load_meta(base, split):
    path = f"{base}/metadata_{split}.json"
    return json.load(open(path)) if os.path.exists(path) else []

def add_paths(samples, base):
    """Attach rgb/fft/hist file paths to each sample dict."""
    for s in samples:
        sid = s["sample_id"]
        s["rgb_path"]  = f"{base}/rgb/{sid}.npy"
        s["fft_path"]  = f"{base}/fft/{sid}.npy"
        s["hist_path"] = f"{base}/hist/{sid}.npy"
    return samples

# Load and annotate each split
train = add_paths(load_meta(FF, "train"), FF) + add_paths(load_meta(GAN, "train"), GAN)
val   = add_paths(load_meta(FF, "val"),   FF) + add_paths(load_meta(GAN, "val"),   GAN)
test  = add_paths(load_meta(FF, "test"),  FF) + add_paths(load_meta(GAN, "test"),  GAN)
celeb = add_paths(load_meta(CELEB, "test"), CELEB)

random.shuffle(train)
random.shuffle(val)

manifest = {"train": train, "val": val, "test": test, "celeb_test": celeb}
json.dump(manifest, open("/kaggle/working/manifest.json", "w"))

print(f"Train   : {len(train):,}")
print(f"Val     : {len(val):,}")
print(f"Test    : {len(test):,}")
print(f"Celeb   : {len(celeb):,}  (cross-dataset, never trained on)")

# Class balance check
for name, split in [("Train", train), ("Val", val)]:
    r = sum(1 for s in split if s["label"] == 0)
    f = sum(1 for s in split if s["label"] == 1)
    print(f"{name}: real={r:,}  fake={f:,}  ratio={r/max(f,1):.2f}")

In [ ]:
# ── CELL 6: Visualize Sample Faces ───────────────────────────
import numpy as np, matplotlib.pyplot as plt, random

real_s = [s for s in train if s["label"] == 0 and os.path.exists(s["rgb_path"])]
fake_s = [s for s in train if s["label"] == 1 and os.path.exists(s["rgb_path"])]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.patch.set_facecolor("#0d1117")

for col, s in enumerate(random.sample(real_s, 4)):
    ax = axes[0][col]
    ax.imshow(np.load(s["rgb_path"]))
    ax.set_title(f"REAL | {s.get('method','')}", color="#2ea043", fontsize=9)
    ax.axis("off")

for col, s in enumerate(random.sample(fake_s, 4)):
    ax = axes[1][col]
    ax.imshow(np.load(s["rgb_path"]))
    ax.set_title(f"FAKE | {s.get('method','')}", color="#f85149", fontsize=9)
    ax.axis("off")

fig.suptitle("Preprocessed Face Crops — 224×224 RGB", color="white", fontsize=13)
plt.tight_layout()
plt.savefig("/kaggle/working/sample_faces.png", dpi=120, facecolor="#0d1117")
plt.show()

In [ ]:
# ── CELL 7: Dataset Balance Chart ────────────────────────────
import matplotlib.pyplot as plt
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor("#0d1117")

for ax in axes:
    ax.set_facecolor("#161b22")
    for sp in ax.spines.values(): sp.set_color("#30363d")
    ax.tick_params(colors="white")

# Real vs Fake
n_r = sum(1 for s in train if s["label"] == 0)
n_f = sum(1 for s in train if s["label"] == 1)
axes[0].pie([n_r, n_f], labels=["Real", "Fake"], colors=["#2ea043","#f85149"],
            autopct="%1.1f%%", textprops={"color":"white"}, wedgeprops={"edgecolor":"#0d1117"})
axes[0].set_title("Class Balance (Train)", color="white", fontsize=12)

# Split sizes
labels = ["Train", "Val", "Test", "Celeb"]
sizes  = [len(train), len(val), len(test), len(celeb)]
bars = axes[1].bar(labels, sizes, color=["#58a6ff","#3fb950","#e3b341","#d2a8ff"], edgecolor="#0d1117")
for bar, v in zip(bars, sizes):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30, f"{v:,}", ha="center", color="white", fontsize=9)
axes[1].set_title("Samples per Split", color="white", fontsize=12)
axes[1].set_ylabel("Count", color="#8b949e")

# Fake method breakdown
methods = Counter(s.get("method","?") for s in train if s["label"] == 1)
axes[2].barh(list(methods.keys()), list(methods.values()), color="#f85149", edgecolor="#0d1117")
axes[2].set_title("Fake Methods (Train)", color="white", fontsize=12)
axes[2].set_xlabel("Count", color="#8b949e")

fig.suptitle("SDP Preprocessed Dataset Overview", color="white", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/dataset_overview.png", dpi=120, facecolor="#0d1117")
plt.show()

In [ ]:
# ── CELL 8: Write model_config.yaml ──────────────────────────
import yaml

config = {
    "model": {
        "ensemble": {
            "cnn_branch": "convnextv2_base",
            "vit_branch": "swinv2_base_window12to16_192to256"
        },
        "pretrained": True,
        "num_classes": 1
    },
    "data": {
        "image_size": 224,          # must match preprocessing output
        "manifest_path": "/kaggle/working/manifest.json",
        "num_workers": 4
    },
    "explainability": {
        "gradcam_layer": "stages.3.blocks.2",
        "ela_quality": 90,
        "fft_shift": True
    },
    "training": {
        "batch_size": 64,           # .npy I/O is fast → large batch is fine
        "learning_rate": 1e-4,
        "epochs": 20,
        "warmup_epochs": 2
    }
}

yaml.dump(config, open("/kaggle/working/model_config.yaml", "w"), default_flow_style=False)
print("✅ model_config.yaml saved")

In [ ]:
# ── CELL 9: Write model.py ────────────────────────────────────
# Defines ConvNextBranch, SwinBranch, and EnsembleDeepfakeDetector.
# Saved to /kaggle/working/ so Notebooks 2 & 3 can import it.

model_py = '''
import torch, torch.nn as nn, timm, yaml, os
from safetensors.torch import save_file, load_file


class ConvNextBranch(nn.Module):
    """ConvNeXt-V2: detects local pixel-level manipulation artifacts."""
    def __init__(self, model_name="convnextv2_base", pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.fc = nn.Linear(self.backbone.num_features, 1)

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features), features


class SwinBranch(nn.Module):
    """Swin-V2: detects global lighting/shadow inconsistencies."""
    def __init__(self, model_name="swinv2_base_window12to16_192to256", pretrained=True):
        super().__init__()
        try:
            self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        except Exception:
            # Fallback if the resolution variant isn't available
            self.backbone = timm.create_model("swinv2_base_window12_256", pretrained=pretrained, num_classes=0)
        self.fc = nn.Linear(self.backbone.num_features, 1)

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features), features


class EnsembleDeepfakeDetector(nn.Module):
    """
    Soft-voting ensemble of ConvNeXt-V2 and Swin-V2.
    Platt Scaling (learnable temperature per branch) calibrates confidence.
    Final logit = average of temperature-scaled branch logits.
    """
    def __init__(self, config_path="/kaggle/working/model_config.yaml"):
        super().__init__()
        cfg = yaml.safe_load(open(config_path))["model"]
        self.cnn_branch = ConvNextBranch(cfg["ensemble"]["cnn_branch"], cfg["pretrained"])
        self.vit_branch = SwinBranch(cfg["ensemble"]["vit_branch"],     cfg["pretrained"])
        self.temperature_cnn = nn.Parameter(torch.ones(1))
        self.temperature_vit = nn.Parameter(torch.ones(1))

    def forward(self, x):
        cnn_logits, _ = self.cnn_branch(x)
        vit_logits, _ = self.vit_branch(x)
        scaled_cnn = cnn_logits / self.temperature_cnn
        scaled_vit = vit_logits / self.temperature_vit
        ensemble   = (scaled_cnn + scaled_vit) / 2.0
        return ensemble, scaled_cnn, scaled_vit

    def save_safetensors(self, path):
        save_file(dict(self.state_dict()), path)
        print(f"Saved → {path}")

    def load_safetensors(self, path):
        self.load_state_dict(load_file(path))
        print(f"Loaded ← {path}")
'''

open("/kaggle/working/model.py", "w").write(model_py.strip())
print("✅ model.py saved")
print("\n→ Notebook 1 complete. Proceed to Notebook 2 for training.")
